# Chapter 6a — Retrieval-Augmented Generation (Hotels)

Companion code for the **first half of Chapter 6** of *Build an Advanced RAG Application (From Scratch)*.

We connect the FAISS retrieval from Chapter 4 to the LLM prompting from Chapter 5 to produce a complete, grounded RAG pipeline:

1. **Retrieve** the most relevant hotel reviews for the user's query.
2. **Augment** the prompt with the retrieved context.
3. **Generate** a Markdown answer with inline citations.

We then graduate from FAISS (vectors live in process memory) to **Qdrant** — a real vector database that supports payloads and metadata filtering (e.g. *only Istanbul hotels*).

> Reusable code: `rag_pipeline.py`, `qdrant_helpers.py`.

## 0. Setup

Needs `OPEN_ROUTER_API_KEY` in your `.env` (sign up at https://openrouter.ai).

In [ ]:
import sys, os
sys.path.insert(0, '.')
sys.path.insert(0, '../chapter_04_semantic_search')

import numpy as np
import torch
from IPython.display import Markdown, display

from data_loader import load_paris_reviews, load_hotel_reviews
from search import load_embedding_model, get_embeddings, build_faiss_cosine_index
from rag_pipeline import generate_answer, search_hotels_by_query
from qdrant_helpers import (
    make_in_memory_client,
    reset_collection,
    upsert_hotel_reviews,
    search_hotels,
)

## 1. Build the index (Paris-only, FAISS)

Same setup as Chapter 4 — we just keep the index alive so we can query it from RAG.

In [ ]:
df_paris = load_paris_reviews()
print(f"Paris reviews: {len(df_paris):,}")

embed_model = load_embedding_model()
if torch.cuda.is_available(): embed_model = embed_model.to("cuda")
elif torch.backends.mps.is_available(): embed_model = embed_model.to("mps")

review_embeddings = embed_model.encode(df_paris["review_text"].tolist(), show_progress_bar=True).astype("float32")
faiss_index = build_faiss_cosine_index(review_embeddings)
print("Index ready.")

## 2. Retrieve only — what FAISS hands to the LLM

Before we generate, look at the raw retrieval. RAG quality is bounded above by retrieval quality.

In [ ]:
query = "Hotel with a view of the Eiffel tower."
results = search_hotels_by_query(query, embed_model, faiss_index, df_paris, k=5)
for r in results:
    print(f"{r['rank']}. {r['hotel_name']}  (cos={r['cosine_similarity']:.3f})")
    print(f"   {r['review_text'][:160]}...\n")

## 3. Full RAG: retrieve → augment → generate

We stream the answer from an OpenRouter-hosted LLM (default: `qwen/qwen3-8b`).

In [ ]:
query = "Hotels with a view of the Eiffel Tower"
answer, sources = generate_answer(query, embed_model, faiss_index, df_paris, k=10)

## 4. Spread to all cities — and add metadata filtering with Qdrant

FAISS gave us fast vector search. But it doesn't know that *some hotels are in Istanbul, others in Paris*. To filter by city we'd have to keep parallel arrays in sync ourselves.

**Qdrant** stores `{vector, payload}` per point and supports server-side filters. That's the production shape.

In [ ]:
# Load all cities, not just Paris
df_all = load_hotel_reviews().drop_duplicates().reset_index(drop=True)
df_all = df_all.dropna(subset=["review_text"]).reset_index(drop=True)
print(f"All-city reviews: {len(df_all):,}")
print(df_all.locality.value_counts().head(10))

In [ ]:
# Embed (this takes a while; restrict for a quick demo)
DEMO_LIMIT = 5000  # set None to embed everything
df_demo = df_all.head(DEMO_LIMIT).reset_index(drop=True) if DEMO_LIMIT else df_all
all_embeddings = embed_model.encode(df_demo["review_text"].tolist(), show_progress_bar=True).astype("float32")
print(all_embeddings.shape)

In [ ]:
qdrant = make_in_memory_client()
COLLECTION = "hotel_reviews"
reset_collection(qdrant, COLLECTION, vector_size=all_embeddings.shape[1])
upsert_hotel_reviews(qdrant, COLLECTION, df_demo, all_embeddings)
print(f"Upserted {len(df_demo):,} reviews into '{COLLECTION}'.")

### City-filtered semantic search

In [ ]:
query = "Amazing hotel close to everything"
city_filter = "Istanbul"

hits = search_hotels(query, embed_model, qdrant, collection_name=COLLECTION, city=city_filter, k=5)
for i, h in enumerate(hits, 1):
    print(f"{i}. {h.payload['hotel_name']}  ({h.payload['locality']})  score={h.score:.3f}")
    print(f"   {h.payload['review_text'][:160]}...\n")

### Full RAG with city filter

Same retrieve → augment → generate flow, but the retrieval is now scoped to one city.

In [ ]:
from rag_pipeline import get_openrouter_client

def generate_answer_qdrant(query, city=None, k=15, llm_model="qwen/qwen3-8b"):
    hits = search_hotels(query, embed_model, qdrant, collection_name=COLLECTION, city=city, k=k)
    context = "\n".join(
        f"Source {i+1}: {h.payload['hotel_name']} ({h.payload['locality']}) — score {h.score:.3f}\n"
        f"  Review: {h.payload['review_text']}"
        for i, h in enumerate(hits)
    )
    prompt = f'''Answer the user's query in Markdown using ONLY the retrieved sources below.
Cite inline as [1][2]. Be concise and concrete; no salutations.

Query: "{query}"
City filter: {city or "(none)"}

Sources:
{context}
'''
    client = get_openrouter_client()
    stream = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    out = ""
    for chunk in stream:
        d = chunk.choices[0].delta.content
        if d:
            print(d, end="", flush=True)
            out += d
    print()
    return out, hits

In [ ]:
answer, sources = generate_answer_qdrant("Amazing hotel close to everything", city="Istanbul", k=10)

## What's next

Notebook **6b** repeats this pipeline on a very different corpus — research-paper abstracts — adding:

- Document chunking (papers can be longer than the embedding model's context window)
- HuggingFace `transformers` directly (skipping `sentence-transformers`)
- Persisting the Qdrant collection to disk